In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from astropy.table import Table

In [2]:
apt_pid = 999
PID_MAP = {981: "GPS", 994: "HLWAS", 998: "GBTDS", 999: "HLTDS"}

In [3]:
joint_df = Table.read("roman_opsim/joint_schedule_apt_pid999.ecsv")
joint_df = joint_df.to_pandas()

In [5]:
# Group by PASS, sorted chronologically
# Each OBSERVATION has multiple VISITs (No filter change; tiling the full field)
pass_groups = [
    grp for _, grp in
    joint_df.sort_values(['mjd_start', 'OBSERVATION', 'VISIT']).groupby('PASS', sort=False)
]

# Find the first pass that has north exposures and the first that has south exposures
first_north_id = next(g['PASS'].iloc[0] for g in pass_groups if g['DEC'].iloc[0] > 0)
first_south_id = next(g['PASS'].iloc[0] for g in pass_groups if g['DEC'].iloc[0] < 0)
expand_ids = {first_north_id, first_south_id}

# Build frame list: expand first north/south passes exposure-by-exposure; keep others as full pass
frame_list = []
for g in pass_groups:
    if g['PASS'].iloc[0] in expand_ids:
        for idx in range(len(g)):
            frame_list.append(g.iloc[[idx]])   # single-row DataFrame
    else:
        frame_list.append(g)

n_frames = len(frame_list)
print(f"Total frames: {n_frames}  (passes: {len(pass_groups)}, "
      f"first north pass {first_north_id}: {len(joint_df[joint_df.PASS==first_north_id])} exposures, "
      f"first south pass {first_south_id}: {len(joint_df[joint_df.PASS==first_south_id])} exposures)")

# --- Filter color palette (wavelength-ordered, colorblind-friendly Paul Tol colors) ---
BP_COLORS = {
    'F062': '#4477AA',
    'F087': '#66CCEE',
    'F106': '#228833',
    'F129': '#CCBB44',
    'F158': '#EE6677',
    'F184': '#AA3377',
    'F213': '#BBBBBB',
    'W146': '#EE8866',
    'PRISM': "#755903",
}

# Filters used in each expanded pass (for legend)
bp_in_exp_n = sorted(joint_df[joint_df['PASS'] == first_north_id]['BANDPASS'].unique())
bp_in_exp_s = sorted(joint_df[joint_df['PASS'] == first_south_id]['BANDPASS'].unique())

north = joint_df[joint_df['DEC'] > 0]
south = joint_df[joint_df['DEC'] < 0]

# Independent x-axes: north on top, south on bottom, panels touch with no gap
fig, (ax_n, ax_s) = plt.subplots(2, 1, figsize=(8, 10))
plt.subplots_adjust(hspace=0)

# North (top): x-axis ticks and label on top; hide bottom spine so boundary is a single line
ax_n.xaxis.tick_top()
ax_n.xaxis.set_label_position('top')
ax_n.spines['bottom'].set_visible(False)
ax_s.spines['top'].set_visible(False)

def setup_ax(ax, df):
    ax.set_ylabel("DEC (deg)")
    ax.set_xlim(df['RA'].max() + 0.5, df['RA'].min() - 0.5)  # RA decreases left→right
    ax.set_ylim(df['DEC'].min() - 0.5, df['DEC'].max() + 0.5)

setup_ax(ax_n, north)
setup_ax(ax_s, south)

ax_n.set_xlabel("RA (deg)")
ax_s.set_xlabel("RA (deg)")

# Textbox labels at top of each subplot (replacing titles)
_box = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='0.7')
ax_n.text(0.5, 0.96, f"PID {apt_pid} ({PID_MAP.get(apt_pid)}) — North",
          transform=ax_n.transAxes, ha='center', va='top', fontsize=10, bbox=_box)
ax_s.text(0.5, 0.96, f"PID {apt_pid} ({PID_MAP.get(apt_pid)}) — South",
          transform=ax_s.transAxes, ha='center', va='top', fontsize=10, bbox=_box)

# Filter legends for the two expanded passes
if bp_in_exp_n:
    patches_n = [mpatches.Patch(color=BP_COLORS.get(bp, 'steelblue'), label=bp)
                 for bp in bp_in_exp_n]
    ax_n.legend(handles=patches_n, loc='lower left', title=f'Pass {first_north_id} filters',
                fontsize=8, title_fontsize=8, framealpha=0.8)
if bp_in_exp_s:
    patches_s = [mpatches.Patch(color=BP_COLORS.get(bp, 'steelblue'), label=bp)
                 for bp in bp_in_exp_s]
    ax_s.legend(handles=patches_s, loc='upper left', title=f'Pass {first_south_id} filters',
                fontsize=8, title_fontsize=8, framealpha=0.8)

scat_n = ax_n.scatter([], [], s=20, alpha=0.5, edgecolors='none')
cur_n  = ax_n.scatter([], [], s=40, alpha=0.9, edgecolors='white', linewidths=0.5)
scat_s = ax_s.scatter([], [], s=20, alpha=0.5, edgecolors='none')
cur_s  = ax_s.scatter([], [], s=40, alpha=0.9, edgecolors='white', linewidths=0.5)

# Time label at the bottom of the south subplot
time_label = ax_s.text(0.5, 0.03, '', transform=ax_s.transAxes,
                        fontsize=9, ha='center', va='bottom')

shown_n_xy, shown_n_cols = [], []
shown_s_xy, shown_s_cols = [], []
EMPTY = np.empty((0, 2))

def update(i):
    if i == 0:
        return scat_n, cur_n, scat_s, cur_s, time_label

    g = frame_list[i - 1]
    is_north = g['DEC'].iloc[0] > 0
    pts = list(zip(g['RA'], g['DEC']))
    pass_id = g['PASS'].iloc[0]
    in_expanded = pass_id in expand_ids

    # Color by filter for the two expanded passes; steelblue/tomato elsewhere
    pt_color  = BP_COLORS.get(g['BANDPASS'].iloc[0], 'steelblue') if in_expanded else 'steelblue'
    cur_color = pt_color if in_expanded else 'tomato'

    if is_north:
        shown_n_xy.extend(pts)
        shown_n_cols.extend([pt_color] * len(pts))
        scat_n.set_offsets(shown_n_xy)
        scat_n.set_facecolor(shown_n_cols)
        cur_n.set_offsets(pts)
        cur_n.set_facecolor([cur_color] * len(pts))
        cur_s.set_offsets(EMPTY)
    else:
        shown_s_xy.extend(pts)
        shown_s_cols.extend([pt_color] * len(pts))
        scat_s.set_offsets(shown_s_xy)
        scat_s.set_facecolor(shown_s_cols)
        cur_s.set_offsets(pts)
        cur_s.set_facecolor([cur_color] * len(pts))
        cur_n.set_offsets(EMPTY)

    if pass_id in expand_ids:
        exp_shown = sum(1 for f in frame_list[:i] if f['PASS'].iloc[0] == pass_id)
        exp_total = len(joint_df[joint_df['PASS'] == pass_id])
        time_label.set_text(
            f"Pass {pass_id} Observation {g['OBSERVATION'].iloc[0]} Visit {g['VISIT'].iloc[0]}  Exposures {exp_shown}/{exp_total}   MJD {g['mjd_start'].iloc[0]:.1f}"
        )
    else:
        time_label.set_text(f"Pass {pass_id}   MJD {g['mjd_start'].iloc[0]:.1f}")

    return scat_n, cur_n, scat_s, cur_s, time_label

plt.rcParams['animation.embed_limit'] = 100  # MB — raised to fit ~52 MB animation
ani = FuncAnimation(fig, update, frames=n_frames + 1, interval=100, blit=True)
plt.close(fig)

#save animation as mp4 (requires ffmpeg)
ani.save(f'animations/pid_{apt_pid}_coverage.mp4', writer='ffmpeg', dpi=150)

Total frames: 871  (passes: 320, first north pass 1: 246 exposures, first south pass 9: 307 exposures)
